# 05 — Per-chunk Context Attribution (debug)

Debug harness for `context_attribution.context_cite`, run the **per-chunk** way:
ContextCite is applied **once per retrieved chunk**, each chunk small enough that
every `prompt_logprobs` call stays under the worker's ~2k ceiling, throttled +
retried so a burst can't crash it.

**What this notebook does and does not do:**

- **No backend API call.** The RAG response is loaded from the local
  `chat_output.txt` mock (a real captured backend payload), so `fetch_backend`
  is never invoked.
- **Still calls the Zingplay vLLM endpoint** for ContextCite (`prompt_logprobs=1`),
  one chunk at a time.

Flow: `load mock → response → per-chunk ContextCite`. (The earlier whole-context
approach — scoring all chunks in one run — was removed: it exceeds the worker's
ceiling. See git history if you need it.)

## Parameters

All knobs for the run live here. Tweak and re-run top-to-bottom.

In [ ]:
# ============================ Parameters ============================

# Mock backend payload to debug against (no live backend call is made).
MOCK_PATH = 'chat_output.txt'
QUERY     = 'Nhân viên VNG được nghỉ phép bao nhiêu ngày mỗi năm?'

# Response cap (chars). The response rides in every prompt_logprobs call, so a
# shorter response leaves more of the per-chunk budget for context.
MAX_RESPONSE_CHARS = 500

# ContextCite ablation settings.
NUM_ABLATIONS      = 32     # number of masked context variants scored per chunk
ABLATION_KEEP_PROB = 0.5    # probability each source is kept in a given ablation
BATCH_SIZE         = 1

# --- Per-chunk pipeline ---
# Run ContextCite once PER CHUNK, each chunk capped so the full-context call stays
# under the worker's ~2k ceiling. Heading + table title-row are fixed scaffolding;
# paragraphs and table data-rows are the ablatable sources.
PER_CHUNK_MAX_CHARS = 2000   # trim a chunk's context above this (keeps calls <= ceiling)
TOP_K               = 3      # how many top sources to print per chunk

# --- Graceful calls: throttle + retry (applied to the module) ---
# The mini worker falls over under bursts of prompt_logprobs calls. Space calls out
# and ride through transient 5xx/connection errors. Operational only — changes no score.
API_CALL_DELAY    = 1.0   # seconds to wait BEFORE each call
API_MAX_RETRIES   = 4     # total attempts per call
API_RETRY_BACKOFF = 3.0   # linear backoff: wait = backoff * attempt_number

# Logging: when True the module logs per-call token counts (and the size/HTTP
# status of any failing call) at INFO.
VERBOSE_API_LOGGING = True

## Setup — import the module and inspect its config

In [ ]:
import sys, json, logging
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))

from dotenv import load_dotenv
load_dotenv(dotenv_path='../.env')

# Import the module under test. It self-configures from .env on import:
# the Zingplay LLM endpoint for prompt_logprobs.
import context_attribution.context_cite as cc_module
from context_attribution.context_cite import prepare_inputs, make_tokenizer, APIModel
from context_cite import ContextCiter

# Response cap from Parameters. Per-chunk uses raw chunks, so we don't truncate the
# (unused) concatenated context — only the response matters for call size.
cc_module.MINI_MAX_RESPONSE_CHARS = MAX_RESPONSE_CHARS
cc_module.MINI_MAX_CONTEXT_CHARS  = None

# Apply graceful-call settings (throttle + retry) to the module's call chokepoint.
cc_module.API_CALL_DELAY    = API_CALL_DELAY
cc_module.API_MAX_RETRIES   = API_MAX_RETRIES
cc_module.API_RETRY_BACKOFF = API_RETRY_BACKOFF

# Logging: keep the root quiet (httpx/transformers stay at WARNING) but turn the
# module's per-call token logging up to INFO so it surfaces in the notebook.
logging.basicConfig(level=logging.WARNING, format='%(levelname)s %(name)s: %(message)s', force=True)
logging.getLogger('context_attribution.context_cite').setLevel(
    logging.INFO if VERBOSE_API_LOGGING else logging.WARNING
)

print('Module under test   :', cc_module.__file__)
print('LLM endpoint (ZP)   :', cc_module.LLM_URL)
print('Model               :', cc_module.MODEL)
print('LLM API key set     :', bool(cc_module.API_KEY))
print('Response cap (chars):', cc_module.MINI_MAX_RESPONSE_CHARS)
print(f'Throttle / retry    : delay={API_CALL_DELAY}s, retries={API_MAX_RETRIES}, backoff={API_RETRY_BACKOFF}s')
print('Verbose API logging :', VERBOSE_API_LOGGING)

# Sanity: we need the Zingplay endpoint for ContextCite, but NOT the backend.
assert cc_module.API_KEY,  'OPENAI_API_KEY missing — ContextCite needs the Zingplay endpoint'
assert cc_module.LLM_URL.startswith('http'), f'LLM_URL looks unset: {cc_module.LLM_URL!r}'
print('\nReady: Zingplay endpoint configured, backend will NOT be called.')

## Health check — is the model backend alive?

A trivial `"hello"` call (`max_tokens=1`) against the Zingplay endpoint, run
*before* any heavy calls. If this 500s, the `local-model-mini` worker is down /
unreachable (e.g. crashed by an earlier oversized prompt) — fix the backend
before going further; no amount of context trimming will help. This mirrors
`context_attribution/tests/test_zp_api.py`.

In [ ]:
import requests

r = requests.post(
    cc_module.LLM_URL,
    headers={'Authorization': f'Bearer {cc_module.API_KEY}'},
    json={
        'model': cc_module.MODEL,
        'messages': [{'role': 'user', 'content': 'hello'}],
        'max_tokens': 1,
        'temperature': 0,
    },
    timeout=30,
)

print(f'POST {cc_module.LLM_URL}')
print(f'model       : {cc_module.MODEL}')
print(f'status      : {r.status_code}')
print(f'body (head) : {r.text[:200]}')

if r.ok:
    print('\nBackend is ALIVE — safe to proceed.')
else:
    print('\n*** Backend is DOWN / unreachable. ***')
    print('A trivial "hello" call failed, so this is a backend problem, not a'
          ' context-size problem. Restart / verify the model worker before running'
          ' the rest of the notebook.')

assert r.ok, f'Model backend not healthy (HTTP {r.status_code}) — fix the endpoint before continuing.'

## Stage 1 — Load mock data (no backend call)

`chat_output.txt` is a captured backend payload (the same shape `fetch_backend`
returns). We load it straight from disk and pair it with the query that produced
it, so the backend is bypassed entirely.

In [ ]:
# MOCK_PATH and QUERY come from the Parameters cell.
# Load directly from file — fetch_backend() is deliberately NOT used here.
with open(MOCK_PATH) as f:
    data = json.load(f)

print('Loaded mock payload :', Path(MOCK_PATH).resolve())
print('Query               :', QUERY)
print('Top-level keys      :', list(data.keys()))
print()
print(f'answer              : {len(data["answer"])} chars')
print(f'knowledge_sources   : {len(data["knowledge_sources"])} chunks')
print(f'mode                : {data.get("mode")}')
print(f'usage               : {data.get("usage")}')
print()
print('--- raw answer ---')
print(data['answer'])
print()
print('--- knowledge source chunks ---')
for i, src in enumerate(data['knowledge_sources']):
    snip = ' '.join(src.get('content', '')[:100].split())
    print(f'  [{i}] score={src.get("score", 0):.3f}  {src.get("document_id", "<no doc>")}')
    print(f'       {snip}...')

## Stage 2 — Response to attribute

`prepare_inputs` applies the response cap and returns the answer we'll attribute.
We only need the **response** here — per-chunk attribution scores it against each
raw chunk from `data`, so the concatenated context it also returns is unused.

In [ ]:
# We only need the (capacity-capped) RESPONSE; the concatenated context is unused
# because per-chunk attribution works from the raw chunks in `data`.
_context, response = prepare_inputs(data)

print(f'Response ({len(response)} chars):')
print(response)
print(f'\nChunks available for per-chunk attribution: {len(data["knowledge_sources"])}')

## Stage 3 — Per-chunk ContextCite with the paragraph/table-row partitioner

Run ContextCite **once per chunk**, each chunk capped at `PER_CHUNK_MAX_CHARS` so
the full-context call stays under the worker's ~2k ceiling, throttled + retried so
a burst can't crash the worker.

**Partitioner (`ParagraphTableContextPartitioner`):**
- **Heading** and a **table's title/separator row** are *fixed scaffolding* —
  always present in every ablation, rendered once, never ablated. This frames each
  piece in its proper context without duplication or credit-leakage.
- **Ablatable sources** are the text **paragraphs** (split on `\n\n`) and the
  table **data rows** (one source each, judged with the column header always shown).

**Caveats to keep in mind** (discussed):
- Per-chunk runs share no common baseline → scores are *not* comparable across
  chunks, and redundant chunks (duplicate tables/boilerplate) get over-credited.
- **Position awareness:** the output prints each source's position index. Watch for
  top scores clustering at low indices (right after the heading) — that would hint
  attribution is tracking *proximity* rather than content.
- Chunks that split to `<2` sources (the short boilerplate) are skipped — nothing
  to ablate.

In [ ]:
import re
import numpy as np
from typing import Optional
from context_cite.context_partitioner import BaseContextPartitioner


def _is_table_block(lines):
    """A markdown table block: header row, a `---` separator row, then data rows."""
    return (len(lines) >= 2 and '|' in lines[0]
            and bool(re.match(r'^[\s|:\-]+$', lines[1])) and '-' in lines[1])


class ParagraphTableContextPartitioner(BaseContextPartitioner):
    """Per-chunk splitter for ContextCite.

    FIXED scaffolding (always present, never ablated, rendered once):
      - the heading line(s)            (markdown '#...')
      - a table's header + separator row
    Ablatable SOURCES:
      - each text paragraph            (blank-line / '\\n\\n' delimited)
      - each table data row            (judged with its column header always shown)

    get_context(mask) always emits the scaffolding plus the kept sources, in
    original order; get_context(all-ones) reproduces the chunk exactly.
    """

    def __init__(self, context: str) -> None:
        super().__init__(context)
        self._cache = {}

    def split_context(self) -> None:
        text = self.context
        tokens = re.split(r'(\n[ \t]*\n)', text)   # even idx = blocks, odd idx = blank-line seps
        segs = []                                   # each: {text, fixed, sep}
        prev_sep = ''
        for i, tok in enumerate(tokens):
            if i % 2 == 1:
                prev_sep = tok
                continue
            if tok == '':
                continue
            lines = tok.split('\n')
            if _is_table_block(lines):
                segs.append({'text': lines[0] + '\n' + lines[1], 'fixed': True, 'sep': prev_sep})
                for ln in lines[2:]:
                    segs.append({'text': ln, 'fixed': False, 'sep': '\n'})
            elif tok.lstrip().startswith('#'):
                segs.append({'text': tok, 'fixed': True, 'sep': prev_sep})   # heading scaffolding
            else:
                segs.append({'text': tok, 'fixed': False, 'sep': prev_sep})  # paragraph source
            prev_sep = ''
        self._cache['segments']   = segs
        self._cache['source_idx'] = [k for k, s in enumerate(segs) if not s['fixed']]

    @property
    def segments(self):
        if self._cache.get('segments') is None:
            self.split_context()
        return self._cache['segments']

    @property
    def _source_idx(self):
        if self._cache.get('source_idx') is None:
            self.split_context()
        return self._cache['source_idx']

    @property
    def num_sources(self) -> int:
        return len(self._source_idx)

    def get_source(self, index: int) -> str:
        return self.segments[self._source_idx[index]]['text']

    def get_context(self, mask: Optional[np.ndarray] = None):
        if mask is None:
            mask = np.ones(self.num_sources, dtype=bool)
        out, first, si = '', True, -1
        for seg in self.segments:
            if not seg['fixed']:
                si += 1
                if not bool(mask[si]):
                    continue
            out += ('' if first else seg['sep']) + seg['text']
            first = False
        return out


# Quick self-check on the loaded data before any API calls.
_p = ParagraphTableContextPartitioner(data['knowledge_sources'][0]['content'])
assert _p.get_context() == data['knowledge_sources'][0]['content'], 'partitioner must round-trip'
print('Partitioner OK. Source counts per chunk (fixed scaffolding excluded):')
for ci, ch in enumerate(data['knowledge_sources']):
    p = ParagraphTableContextPartitioner(ch['content'][:PER_CHUNK_MAX_CHARS])
    print(f'  chunk {ci}: {len(ch["content"]):>4} chars -> {p.num_sources} ablatable sources')

In [ ]:
# Per-chunk ContextCite. One faithful run per chunk; every call <= PER_CHUNK_MAX_CHARS,
# throttled + retried via the module settings applied in Setup.
per_chunk_results = {}

for ci, ch in enumerate(data['knowledge_sources']):
    content = ch['content'][:PER_CHUNK_MAX_CHARS]
    part = ParagraphTableContextPartitioner(content)
    heading = next((s['text'] for s in part.segments if s['fixed']), '<no heading>')

    print('=' * 78)
    print(f'CHUNK {ci}  | score={ch.get("score", 0):.3f} | {ch.get("document_id", "")[:60]}')
    print(f'  heading : {heading[:70]}')
    print(f'  sources : {part.num_sources} ablatable')

    if part.num_sources < 2:
        print('  -> <2 sources, nothing to ablate. Skipped.')
        per_chunk_results[ci] = None
        continue

    cc = ContextCiter(
        model=APIModel(response=response),
        tokenizer=make_tokenizer(),
        context=content,
        query=QUERY,
        num_ablations=NUM_ABLATIONS,
        ablation_keep_prob=ABLATION_KEEP_PROB,
        batch_size=BATCH_SIZE,
        partitioner=part,
    )
    try:
        attrs = cc.get_attributions(as_dataframe=False, verbose=False)
    except Exception as e:
        print(f'  -> FAILED: {e.__class__.__name__}: {str(e)[:160]}')
        per_chunk_results[ci] = None
        continue

    per_chunk_results[ci] = (part, attrs)
    order = np.argsort(attrs)[::-1]
    print(f'  top {min(TOP_K, len(order))} sources (pos = position index; watch for low-pos clustering):')
    for idx in order[:TOP_K]:
        snippet = part.get_source(int(idx)).replace('\n', ' / ')[:64]
        print(f'      pos[{int(idx)}]  score={attrs[int(idx)]:+.4f}   {snippet}')

print('\nDone. per_chunk_results[chunk_index] = (partitioner, attribution_array) or None.')